#### LLM Based Parallel workflow that takes an Essay and Evaluates its Language quality, Depth of Analysis and Clarity of Thought with thier respective scores in parallel

#### Summarizes it to an overall feedback and an overall score.

In [ ]:
# required imports
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [80]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-120b")

In [68]:
# Output schema
class EvaluationSchema(BaseModel):

    feedback: str = Field(description="detailed feedback on the essay")
    score: int = Field(description="score out of 10", ge=0, le=10)

In [69]:
structured_model = model.with_structured_output(EvaluationSchema)

In [70]:
essay = """Title: The Structural Imperatives of Addressing Unemployment in Pakistan’s Evolving Economy

The escalating unemployment rate in Pakistan represents one of the most formidable macroeconomic challenges facing the nation in the mid-2020s. This phenomenon is not merely a byproduct of temporary market fluctuations but is deeply rooted in structural imbalances, demographic pressures, and a misalignment between educational outputs and industry requirements. As the labor force continues to expand, the inability of the formal sector to absorb new entrants poses a significant threat to social stability and long-term economic growth.

A primary driver of this crisis is the "youth bulge." With a significant percentage of the population under the age of 30, Pakistan possesses a potential demographic dividend that is currently transitioning into a demographic burden. The domestic economy, hampered by inconsistent GDP growth and high inflation, has struggled to create the roughly 1.3 million jobs required annually to keep pace with this growth. Furthermore, the industrial sector has faced headwinds due to energy costs and limited foreign direct investment, leading to a contraction in manufacturing—a traditional engine for mass employment.

Moreover, there is a critical "skills gap" within the labor market. While higher education enrollment has increased, the curriculum often remains detached from the digital and technical competencies demanded by the globalized "Fourth Industrial Revolution." Addressing this requires a strategic shift toward vocational training, digital literacy, and fostering an ecosystem conducive to entrepreneurship. Ultimately, solving Pakistan’s unemployment crisis necessitates a multi-faceted approach: stabilizing the macroeconomic environment, incentivizing the private sector, and overhauling the educational framework to ensure that the workforce is not just educated, but employable."""

essay2 = """Title: Why many peple dont have jobs in Pakistan

Today in Pakistan many young boys and girls are finishing their study but they are sitting at home because they dont have any job. This is very big problem for our country. When we go to the market or see on the news, everyone is talking about how hard it is to find work. The unployement rate is going up very fast and it is making peple very sad and worried for their future.

One big reason for this is that there is no factory or big business opening in our cities. When there is no new business, how can we get jobs? Also, the things in the shops are very expensive, so peple dont have money to spend, and this makes the economy very slow. Many students go to collage but when they come out, the boss says you dont have experience or you dont know how to use new computer software. This is very unfair for the students who worked hard for many years.

In my opinion, the goverment should help the poor peple and start new projects like building roads or schools so peple can get work. If we dont give jobs to our youth, they will go to other country like Dubai or Europe to find work. We need to fix our country and make sure that every person who wants to work can find a good place to earn money for their family. If we work together, maybe the situation will become better soon."""

In [54]:
prompt = f"Evaluate the language quality of the following essay, provide a feedback and assign a score in whole numbers out of 10: {essay}"

In [55]:
# testing the schema and model response
structured_model.invoke(prompt).feedback

'The essay demonstrates a strong command of academic English. The introduction clearly frames the issue, and the subsequent paragraphs develop the argument with appropriate terminology and logical flow. Vocabulary is varied and precise (e.g., “structural imbalances,” “demographic dividend,” “Fourth Industrial Revolution”), and the use of transitional phrases helps maintain coherence. Sentence structures are generally well‑crafted, with a good mix of complex and concise statements. Minor weaknesses include occasional redundancy (e.g., repeating the notion of a “demographic burden”) and a tendency toward overly formal phrasing that can affect readability for a broader audience. Additionally, a few commas could be better placed for smoother pacing. Overall, the language is articulate, cohesive, and suitably formal for the topic.'

In [ ]:
# defining the state for the workflow
class CSS_State(TypedDict):

    essay: str
    lang_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    overall_score: float

In [ ]:
# function to evaluate language quality and assign laguage score
def evaluate_language(state: CSS_State):

    prompt = f"Evaluate the language quality of the following essay, provide a feedback and assign a score in whole numbers out of 10: \n{state['essay']}"
    response = structured_model.invoke(prompt)

    return { 'lang_feedback' : response.feedback, 'individual_scores' : [response.score] }

In [ ]:
# function to evaluate depth of analysis and assign analysis score
def evaluate_analysis(state: CSS_State):

    prompt = f"Evaluate the depth of analysis of the following essay, provide a feedback and assign a score in whole numbers out of 10: \n{state['essay']}"
    response = structured_model.invoke(prompt)

    return { 'analysis_feedback' : response.feedback, 'individual_scores' : [response.score] }

In [ ]:
# function to evaluate clarity of thought and assign clarity score
def evaluate_clarity(state: CSS_State):

    prompt = f"Evaluate the clarity of thought of the following essay, provide a feedback and assign a score in whole numbers out of 10: \n{state['essay']}"
    response = structured_model.invoke(prompt)

    return { 'clarity_feedback' : response.feedback, 'individual_scores' : [response.score] }

In [ ]:
# function to generate final summarized feedback and overall score
def final_evaluation(state: CSS_State):

    prompt = f"Generate a summarized feedback based on the following feedbacks: \nLanguage Quality Feedback: {state['lang_feedback']} \nDepth of Analysis Feedback: {state['analysis_feedback']} \nClarity of Thought Feedback: {state['clarity_feedback']}"
    summarized_feedback = model.invoke(prompt).content

    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback' : summarized_feedback, 'overall_score' : avg_score}

In [71]:
# defining the state graph for the workflow
graph = StateGraph(CSS_State)

# nodes
graph.add_node("Evaluate_Language", evaluate_language)
graph.add_node("Evaluate_Analysis", evaluate_analysis)
graph.add_node("Evaluate_Clarity", evaluate_clarity)
graph.add_node("Final_Evaluation", final_evaluation)

# edges
graph.add_edge(START, 'Evaluate_Language')
graph.add_edge(START, 'Evaluate_Analysis')
graph.add_edge(START, 'Evaluate_Clarity')

graph.add_edge('Evaluate_Language', 'Final_Evaluation')
graph.add_edge('Evaluate_Analysis', 'Final_Evaluation')
graph.add_edge('Evaluate_Clarity', 'Final_Evaluation')

graph.add_edge('Final_Evaluation', END)

In [72]:
workflow = graph.compile()

In [75]:
initial_state1 = {
    'essay' : essay
}

In [76]:
initial_state2 = {
    'essay' : essay2
}

In [78]:
workflow.invoke(initial_state1)

{'essay': 'Title: The Structural Imperatives of Addressing Unemployment in Pakistan’s Evolving Economy\n\nThe escalating unemployment rate in Pakistan represents one of the most formidable macroeconomic challenges facing the nation in the mid-2020s. This phenomenon is not merely a byproduct of temporary market fluctuations but is deeply rooted in structural imbalances, demographic pressures, and a misalignment between educational outputs and industry requirements. As the labor force continues to expand, the inability of the formal sector to absorb new entrants poses a significant threat to social stability and long-term economic growth.\n\nA primary driver of this crisis is the "youth bulge." With a significant percentage of the population under the age of 30, Pakistan possesses a potential demographic dividend that is currently transitioning into a demographic burden. The domestic economy, hampered by inconsistent GDP growth and high inflation, has struggled to create the roughly 1.3 

In [79]:
workflow.invoke(initial_state2)

{'essay': 'Title: Why many peple dont have jobs in Pakistan\n\nToday in Pakistan many young boys and girls are finishing their study but they are sitting at home because they dont have any job. This is very big problem for our country. When we go to the market or see on the news, everyone is talking about how hard it is to find work. The unployement rate is going up very fast and it is making peple very sad and worried for their future.\n\nOne big reason for this is that there is no factory or big business opening in our cities. When there is no new business, how can we get jobs? Also, the things in the shops are very expensive, so peple dont have money to spend, and this makes the economy very slow. Many students go to collage but when they come out, the boss says you dont have experience or you dont know how to use new computer software. This is very unfair for the students who worked hard for many years.\n\nIn my opinion, the goverment should help the poor peple and start new projec